> **This notebook is the walkthrough, not the source of truth.**
> It explains the pipeline stage by stage, in reading order. The code that
> actually runs in the deployed web app lives in
> [`../backend/pipeline.py`](../backend/pipeline.py). The architecture is the
> same and most functions are identical; `pipeline.py` additionally carries a
> few small robustness fixes kept out of the notebook so it stays a clean read:
> grounding-metadata title fallback, store-over-blog price preference, a
> sentinel-vs-offer conflict rule, and optional candidate concurrency. See
> [`../agent/README.md`](../agent/README.md) for the details.

# Lipstick Dupe Price-Finder Agent

## What this notebook does

When two lipstick shades are perceptually identical — their color distance (Delta E) is below 3,
the threshold where the human eye can no longer reliably tell them apart — color stops being a
useful discriminator. The best remaining signal is **price**. This notebook builds an agent
that takes a **tie set** (a list of perceptually equivalent candidates) and finds the cheapest
one that's currently in stock in the user's country.

This is the downstream half of a dupe-finder app. The upstream half (Delta E ranking against a
full shade catalog) already exists; this agent handles the secondary ranking once color
similarity saturates.

## Architecture

The pipeline has three stages:

```
[Input: tie set of candidates + country]
        ↓
① search_agent  google_search("MAC Lipstick Ruby Woo buy price USD")
                → reads Google Shopping result cards
                → returns snippet prices + grounded product URLs

   Python: find entries where price=null but url is present

   fetch_agent  fetch(url)  ← MCP tool (mcp-server-fetch)
                → visits product pages where snippet had a URL but no price
                → reads the actual price from the page HTML

                → combined JSON array: [{retailer, price, currency, in_stock, url}, ...]
        ↓  (two-agent pipeline per candidate, candidates run sequentially)
② Python optimize(all_priced_candidates)
         → deterministic min() over prices
         → flags tradeoff if cheapest ≠ closest color (and gap > 15%)
        ↓
③ Python render_results(outcome)
         → HTML cards: recommendation + optional tradeoff alternative
```

**Why two agents?** Gemini's built-in `google_search` tool and function-calling tools (MCP)
cannot coexist in the same request — the API rejects it with `INVALID_ARGUMENT`. Two separate
agents, each with one tool type, solve this cleanly.

**Why Python for `optimize()`?** LLMs are unreliable at `min()` and tie-breaking.
Deterministic Python is not.

## ADK concepts demonstrated

| Concept | Where |
|---|---|
| **ADK agents** | Two `LlmAgent` instances (`search_agent`, `fetch_agent`), `InMemoryRunner`, full ADK async event loop |
| **MCP server** | `McpToolset` + `StdioConnectionParams` → `mcp-server-fetch` subprocess |
| **Grounding** | `google_search` (ADK built-in) — live Google Search + Shopping panels |
| **Security** | Keys from env vars only; V5 grep cell confirms no hardcoded secrets |
| **Multi-agent** | Python orchestration sequences two specialized agents per candidate |

## Prerequisites

- `GOOGLE_API_KEY` in your `.env` file (loaded by `adk_setup.py`)
- Python 3.11+, `google-adk`, `google-genai`, `mcp`, `mcp-server-fetch`

In [247]:
# Cell 0 — install dependencies (run once)
%pip install -q mcp mcp-server-fetch


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [248]:
# Cell 0b — ADK setup (loads GOOGLE_API_KEY, imports)
%run adk_setup.py

✅ Gemini API key setup complete.
✅ ADK setup complete.


In [249]:
# Cell 0c — model config (change MODEL here to switch everywhere)
MODEL = "gemini-2.5-flash"

## 1. Stub tie set

### What is a tie set?

A tie set is the output of the upstream Delta E color-matching step. Delta E (ΔE) is a
perceptual color distance metric: ΔE < 1 is imperceptible, ΔE < 3 is indistinguishable
to most people under normal viewing conditions. When several candidate dupes all fall
below ΔE 3, they are "tied" on color — and price becomes the tiebreaker.

In production this list is generated automatically from the shade catalog. Here we stub
it with four well-known red lipsticks at a spread of price points (drugstore → luxury)
so we can develop and test the agent with realistic data.

Each entry needs:
- `brand`, `product`, `shade` — used to build the search query
- `delta_e` — color distance to the target shade; used by `optimize()` to detect tradeoffs

In [250]:
# Cell 1 — tie set (stub input from upstream Delta E ranker)
TIE_SET = [
    {
        "brand": "NYX",
        "product": "Soft Matte Lip Cream",
        "shade": "Monte Carlo",
        "delta_e": 1.4,
    },
    {
        "brand": "ColourPop",
        "product": "Lux Liquid Lip",
        "shade": "First Move",
        "delta_e": 2.1,
    },
    {
        "brand": "MAC",
        "product": "Lipstick",
        "shade": "Ruby Woo",
        "delta_e": 0.8,
    },
    {
        "brand": "Charlotte Tilbury",
        "product": "Matte Revolution Lipstick",
        "shade": "Red Carpet Red",
        "delta_e": 2.7,
    },
]

print(f"Tie set: {len(TIE_SET)} candidates (all ΔE < 3 from target)")
for c in TIE_SET:
    print(f"  ΔE={c['delta_e']:.1f}  {c['brand']} — {c['product']} in {c['shade']}")

Tie set: 4 candidates (all ΔE < 3 from target)
  ΔE=1.4  NYX — Soft Matte Lip Cream in Monte Carlo
  ΔE=2.1  ColourPop — Lux Liquid Lip in First Move
  ΔE=0.8  MAC — Lipstick in Ruby Woo
  ΔE=2.7  Charlotte Tilbury — Matte Revolution Lipstick in Red Carpet Red


## 2. Country configuration

### How country detection works in production

In a deployed web app the country comes from the user's IP address. A GeoIP lookup
(e.g. `ipapi`, `geoip2`) maps the request IP to a two-letter country code, which is
passed directly into `run_tie_set(tie_set, country="UK")`. `_COUNTRY_HINTS` then maps
that code to a localized search phrase appended to the query:

```
IP address → GeoIP → "UK" → _COUNTRY_HINTS["UK"] → "buy price GBP"
```

The full search query the agent sends becomes:
```
"MAC Lipstick Ruby Woo buy price USD"          # US
"MAC Lipstick Ruby Woo buy price GBP"          # UK
"MAC Lipstick Ruby Woo comprar preço EUR"      # PT
```

This steers Google Shopping toward local retailers and the right currency without any
additional filtering or per-country logic in the agent.

### Why this demo runs US only

Two practical reasons:
- Most products in the tie set are widely available in the US; UK/EU availability
  is spottier and harder to verify.
- Error analysis (checking whether the agent found real prices vs. hallucinated URLs)
  is much easier when we can manually check US retailers like Ulta, Sephora, and CVS.

The multi-country infrastructure is in place — `_COUNTRY_HINTS` and the `country`
parameter flow all the way through `run_tie_set` → `price_one_candidate` → the search
query. V4 in the verification section exercises a UK run to confirm the wiring works.

In [251]:
# Cell 2 — country hints (IP → country code → localized search phrase)
import re
import json
import asyncio

_COUNTRY_HINTS = {
    "US": "buy price USD",
    "UK": "buy price GBP",
    "PT": "comprar preço EUR",
}

print("Country hints:", list(_COUNTRY_HINTS.keys()))

Country hints: ['US', 'UK', 'PT']


## 3. Price-finder agents

### Two-agent pipeline

Gemini's built-in `google_search` tool and function-calling tools (like MCP) cannot
coexist in the same API request — the model returns `INVALID_ARGUMENT: Built-in tools
and Function Calling cannot be combined`. The solution is two specialized agents that
run in sequence, coordinated by the Python orchestration layer:

```
Agent 1 — search_agent  (google_search only)
  ↓  Queries live Google including Shopping panels
  ↓  Returns retailer names, snippet prices, and grounded product URLs
  ↓  Fast: one call often surfaces 3–5 retailers at once

Python: parse JSON, identify price=null entries that have a URL

Agent 2 — fetch_agent  (MCP fetch only)
  ↓  Called once per null-price URL returned by search_agent
  ↓  Uses mcp-server-fetch to visit the product page
  ↓  Extracts the price from the page HTML
  ↓  Converts null prices into real prices
```

**Why two agents?** Google Shopping snippets show prices for some retailers but not
all — cvs.com and ulta.com frequently appear in results without a price number.
When `search_agent` returns a grounded URL with no price, `fetch_agent` visits that
URL and reads the actual price from the page.

### MCP server

`mcp-server-fetch` is a Python MCP server (`pip install mcp-server-fetch`). `fetch_agent`
connects to it via `McpToolset` with `StdioConnectionParams` — ADK spawns the server
as a subprocess using `sys.executable`, communicates over stdin/stdout, and discovers
the `fetch` tool automatically. No Node.js required.

### Security

Both agents run on the same `GOOGLE_API_KEY` loaded by `adk_setup.py`.
`mcp-server-fetch` makes direct HTTP GET requests — no additional credentials needed.
`fetch_agent` only receives URLs returned by `search_agent`; the orchestration code
never constructs or guesses URLs.

### What the pipeline returns

A merged JSON array — one entry per retailer:
```json
[
  {"retailer": "ulta.com",  "price": 27.0, "currency": "USD",
   "in_stock": true, "url": "https://...", "confidence": "high"},
  {"retailer": "macys.com", "price": 23.0, "currency": "USD",
   "in_stock": true, "url": "https://...", "confidence": "high"}
]
```

In [252]:
# Cell 3 — price-finder agents (search_agent + fetch_agent)
import sys

# ── Agent 1: search_agent ─────────────────────────────────────────────────────
# google_search is a Gemini built-in tool. Built-in tools cannot be combined
# with function-calling tools (like MCP) in one request, so search and fetch
# live in separate agents coordinated by Python orchestration.
SEARCH_INSTRUCTION = """
You are a lipstick price comparison specialist. Find current online retail prices.

## RULES
- NEVER state a price from training memory. Prices change daily — always search first.
- Include every retailer where a price is shown or a product URL is available.
- NEVER construct or guess a URL. Only use URLs that appear in the search results.
  A root domain like "https://www.ulta.com" is not a product URL — return null instead.

## Steps
1. Search for the product using the query provided.
2. Read each result — Google Shopping results show the price directly,
   e.g. "MAC Lipstick Ruby Woo — $23.00 · In stock".
3. Collect every retailer name, price, stock status, and URL from the search.

## Output format
Return ONLY a JSON array — no prose, no markdown fences, no explanation:
[
  {"retailer": "sephora.com", "price": 28.0, "currency": "USD",
   "in_stock": true, "url": "https://...", "confidence": "high"}
]
- price: numeric value, or null if not visible in snippet
- url: exact URL from the search result, null if not present
- confidence: REQUIRED — always include. "high" = price clearly shown, "medium" = inferred, "low" = uncertain

## Error cases
If no Shopping results or product listings are found, return a single-element array
with a status field instead of an empty array:
[{"retailer": null, "price": null, "currency": null, "in_stock": false,
  "url": null, "confidence": "none", "status": "not_found"}]

Status values:
- "not_found"    — no results found for this product
- "discontinued" — search results indicate the product has been discontinued or delisted

For normal results, omit the status field.
"""

search_agent = LlmAgent(
    name="lipstick_search",
    model=Gemini(model=MODEL),
    instruction=SEARCH_INSTRUCTION,
    tools=[google_search],
)

# ── Agent 2: fetch_agent ──────────────────────────────────────────────────────
# MCP fetch only. Python orchestration calls this once per null-price URL
# that search_agent returned.
mcp_fetch_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,          # same interpreter as this kernel
            args=["-m", "mcp_server_fetch", "--ignore-robots-txt"],
        ),
        timeout=30,
    )
)

FETCH_INSTRUCTION = """
You are a product identity + price extractor. Given a product page URL and the product
we are looking for, fetch the page ONCE and extract both what the page is and its price.

## Fetch rules (important — controls cost and reliability)
- Call the fetch tool EXACTLY ONCE for the URL.
- Do NOT pass raw=true. The default simplified text is what you want — raw HTML is
  mostly CSS/JS noise with no price.
- Do NOT paginate with start_index. Instead request enough text in one call by passing
  a large max_length (for example max_length=20000).
- The simplified text begins with the page <title>, which normally names the brand and
  shade — use it to identify the product.

## Steps
1. fetch(url, max_length=20000)   # one call, simplified text
2. From that text read: the product title, brand, shade / color name, price, stock status.

## Output format
Return ONLY a JSON object — no prose, no markdown fences:
{"title": "...", "brand": "...", "shade": "...", "price": 28.0, "currency": "USD", "in_stock": true}
- title:    the product page title, verbatim (usually the first line), or null
- brand:    brand shown on the page, or null
- shade:    shade / color name shown on the page, or null
- price:    numeric value, or null if not found on the page
- currency: "USD", "GBP", "EUR", etc.
- in_stock: true / false / null if unknown

Never invent values. If the page failed to load or shows an error, return all fields null.
"""

fetch_agent = LlmAgent(
    name="lipstick_fetcher",
    model=Gemini(model=MODEL),
    instruction=FETCH_INSTRUCTION,
    tools=[mcp_fetch_toolset],
)

print(f"search_agent: {search_agent.name}  |  model: {MODEL}")
print(f"fetch_agent:  {fetch_agent.name}  |  model: {MODEL}")
print("Pipeline: google_search → [MCP fetch null-price URLs] → merged prices")

search_agent: lipstick_search  |  model: gemini-2.5-flash
fetch_agent:  lipstick_fetcher  |  model: gemini-2.5-flash
Pipeline: google_search → [MCP fetch null-price URLs] → merged prices


## 4. `optimize()` — deterministic price ranker

### Why Python, not the LLM

Once all candidates are priced, choosing the winner is a simple `min()` with a few edge
cases. LLMs are surprisingly unreliable at this: they hallucinate order, miscalculate
percentages, and are inconsistent at tie-breaking. Plain Python is not. Any step that can
be expressed deterministically should be.

### The tradeoff problem

The closest color match (smallest ΔE) and the cheapest option are often different products.
`optimize()` detects this and surfaces both when the gap is meaningful (>15% price
difference), so the user can make an informed choice rather than the system silently
picking one.

Example:
- MAC Ruby Woo (ΔE=0.8) at $22 — closest color
- NYX Monte Carlo (ΔE=1.4) at $8 — cheapest, but perceptibly different?

At ΔE < 3 the human eye can't reliably tell the difference, so the "closest" advantage
is theoretical. But surfacing it lets the user decide.

### `strategy` parameter

- `"cheapest"` (default) — returns the lowest-priced in-stock candidate; if the
  closest-color option costs >15% more, it's shown as an alternative.
- `"closest"` — returns the smallest-ΔE in-stock candidate; if it costs >15% more
  than the cheapest, the cheapest is shown as an alternative.

The three unit tests below verify the three cases: no tradeoff, tradeoff flagged,
and no in-stock candidates.

In [253]:
# Cell 4 — optimize() + unit tests
def optimize(priced_candidates: list[dict], strategy: str = "cheapest") -> dict:
    """
    Deterministically pick the best candidate from a priced tie set.

    Args:
        priced_candidates: Each dict has brand/product/shade/delta_e plus a "prices"
            list of {retailer, price, currency, in_stock, url, confidence} dicts.
        strategy: "cheapest" (default) — lowest price among in-stock options.
                  "closest"           — smallest delta_e (closest color match).

    Returns:
        {
            "recommendation": chosen candidate dict (with _best_price/_best_retailer attached),
            "retailer":       the cheapest in-stock retailer entry for the recommendation,
            "alternative":    alternative candidate when a meaningful price tradeoff exists,
            "tradeoff":       True when cheapest ≠ closest AND price gap exceeds 15%,
        }
    """
    def _best_in_stock(candidate: dict) -> tuple[float | None, dict | None]:
        options = [
            p for p in candidate.get("prices", [])
            if p.get("in_stock") and p.get("price") is not None
        ]
        if not options:
            return None, None
        best = min(options, key=lambda p: p["price"])
        return best["price"], best

    scored = []
    for c in priced_candidates:
        val, retailer = _best_in_stock(c)
        if val is not None:
            scored.append({**c, "_best_price": val, "_best_retailer": retailer})

    if not scored:
        return {
            "recommendation": None, "retailer": None,
            "alternative": None, "tradeoff": False,
            "note": "No in-stock candidates found.",
        }

    by_price = min(scored, key=lambda c: c["_best_price"])
    by_color = min(scored, key=lambda c: c.get("delta_e", float("inf")))

    chosen = by_price if strategy == "cheapest" else by_color

    # Tradeoff: cheapest shade ≠ closest-color shade AND gap exceeds 15%
    different = by_price.get("shade") != by_color.get("shade")
    gap = (
        (by_color["_best_price"] - by_price["_best_price"]) / by_color["_best_price"]
        if by_color["_best_price"] > 0 else 0
    )
    tradeoff = different and gap > 0.15

    alternative = None
    if tradeoff:
        alternative = by_color if strategy == "cheapest" else by_price

    return {
        "recommendation": chosen,
        "retailer":       chosen["_best_retailer"],
        "alternative":    alternative,
        "tradeoff":       tradeoff,
    }


# ── Unit tests ────────────────────────────────────────────────────────────────

def _make(shade, delta_e, price, in_stock=True):
    return {
        "brand": "TestBrand", "product": "Lip", "shade": shade, "delta_e": delta_e,
        "prices": [{"retailer": "test.com", "price": price, "currency": "USD",
                    "in_stock": in_stock, "url": "https://test.com", "confidence": "high"}],
    }

_tests = [
    # strategy="cheapest" (default)
    (optimize([_make("A", 0.5, 12.0), _make("B", 2.0, 30.0)]),
     "A", False, "cheapest: cheapest wins, no tradeoff (cheapest == closest)"),
    (optimize([_make("A", 0.5, 40.0), _make("B", 2.0, 12.0)]),
     "B", True,  "cheapest: tradeoff flagged (cheapest ≠ closest, gap >15%)"),
    (optimize([_make("A", 0.5, 30.0, in_stock=False)]),
     None, False, "cheapest: no in-stock candidates"),
    # strategy="closest"
    (optimize([_make("A", 0.5, 40.0), _make("B", 2.0, 12.0)], strategy="closest"),
     "A", True,  "closest: closest wins, tradeoff flagged (it costs >15% more)"),
    (optimize([_make("A", 0.5, 12.0), _make("B", 2.0, 30.0)], strategy="closest"),
     "A", False, "closest: closest wins, no tradeoff (it is also cheapest)"),
    (optimize([_make("A", 0.5, 30.0, in_stock=False)], strategy="closest"),
     None, False, "closest: no in-stock candidates"),
]

for result, exp_shade, exp_tradeoff, label in _tests:
    got_shade    = result["recommendation"]["shade"] if result["recommendation"] else None
    got_tradeoff = result["tradeoff"]
    ok = got_shade == exp_shade and got_tradeoff == exp_tradeoff
    print(f"{'✓' if ok else '✗'} {label}: shade={got_shade}, tradeoff={got_tradeoff}")

✓ cheapest: cheapest wins, no tradeoff (cheapest == closest): shade=A, tradeoff=False
✓ cheapest: tradeoff flagged (cheapest ≠ closest, gap >15%): shade=B, tradeoff=True
✓ cheapest: no in-stock candidates: shade=None, tradeoff=False
✓ closest: closest wins, tradeoff flagged (it costs >15% more): shade=A, tradeoff=True
✓ closest: closest wins, no tradeoff (it is also cheapest): shade=A, tradeoff=False
✓ closest: no in-stock candidates: shade=None, tradeoff=False


## 5. Orchestration: tie set → price each → optimize

### Two-agent pipeline per candidate

For each candidate the orchestration runs two agents in sequence:

```
① search_agent (google_search)
   "NYX Soft Matte Lip Cream Monte Carlo buy price USD"
   → Shopping snippet prices for 3–5 retailers + grounded URLs
   → Some retailers appear with a URL but price=null (CVS, Ulta)
   → Some appear with a price but low/medium confidence (Walmart marketplace)

② Python: parse JSON, identify entries to fetch

③ fetch_agent (MCP fetch) — one call per entry that needs verification
   → null-price URLs: visits page to get the actual price
   → low/medium-confidence prices: verifies the snippet price is real
     If the page says unavailable → clears the snippet price (won't win in optimize)
     If the page confirms a price → updates with high confidence

→ Merged JSON array of verified retailer entries

optimize(all_priced_candidates) → recommendation
```

### Why verify low-confidence snippet prices?

Google Shopping snippets can surface a price for a Walmart listing that is technically
available — but only through a marketplace seller at a much higher price, or listed as
unavailable. The snippet shows the base price; the actual page tells a different story.
By fetching those pages, the agent catches this and removes bad prices before `optimize()`
picks a winner based on stale data.

**Tradeoff:** more `fetch_agent` calls per candidate (1–3 extra). Each MCP fetch adds
~2–5 seconds and a small token cost. For a tie set of 2–4 candidates with 3–5 retailers
each this is acceptable; for larger sets consider raising the confidence threshold.

### Sequential execution

Candidates run sequentially to stay within the free-tier RPM limit. A 429 retry is in
place with `retryDelay` backoff. Each agent call uses its own `InMemoryRunner` and
session so state doesn't bleed between candidates or between pipeline stages.

### What `run_tie_set` returns

```python
{
    "recommendation": {..., "_best_price": 8.99, "_best_retailer": {...}},
    "retailer":       {"retailer": "ulta.com", "price": 8.99, "currency": "USD",
                       "in_stock": True, "url": "...", "confidence": "high"},
    "alternative":    {...},   # None if no meaningful tradeoff
    "tradeoff":       True,
    "all_candidates": [...],
}
```

## 5a. Trace logging

Every ADK event from `runner.run_async()` is captured and written to a JSONL file —
one file per candidate. This lets you inspect what the agent did without cluttering
the notebook output.

### What events appear in a trace

The agent has one tool: `google_search`. A typical trace for one candidate:

```
{"type": "tool_call",     "name": "google_search", "content": {"query": "MAC Lipstick Ruby Woo buy price USD"}}
{"type": "tool_response", "name": "google_search", "content": {"result": "Shopping results: Sephora $28, Ulta $27.50 ..."}}
{"type": "text",          "name": null,            "content": "[{\"retailer\": \"sephora.com\", \"price\": 28.0, ...}]"}
```

If no Shopping cards are visible for the query (e.g. a very niche brand), you'll see the
search response but the final text may return empty or low-confidence entries.

### `RunTracer` class

One tracer per candidate. Accumulates events via `.record(event)`, writes the JSONL
file on `.flush()`. The filename encodes the timestamp and candidate label.

In [254]:
# Cell 5a — RunTracer (writes per-candidate JSONL trace files)
import json
from datetime import datetime, timezone
from pathlib import Path

TRACE_DIR = Path("traces")
TRACE_DIR.mkdir(exist_ok=True)


def _event_to_records(event, candidate_label: str) -> list[dict]:
    """Flatten one ADK event into a list of JSON-serializable trace records."""
    ts     = datetime.now(timezone.utc).isoformat()
    author = getattr(event, "author", "unknown")
    role   = getattr(event.content, "role", None) if event.content else None
    base   = {"ts": ts, "candidate": candidate_label, "author": author, "role": role}

    records = []

    # Grounding metadata: the authoritative (domain, title, uri) triples that
    # google_search actually used. Captured independently of the model's JSON so
    # we can recover product URLs the model failed to transcribe into its answer.
    gm = getattr(event, "grounding_metadata", None)
    if gm and getattr(gm, "grounding_chunks", None):
        chunks = []
        for ch in gm.grounding_chunks:
            web = getattr(ch, "web", None)
            if web and (web.uri or web.domain or web.title):
                chunks.append({"domain": web.domain, "title": web.title, "uri": web.uri})
        if chunks:
            records.append({**base, "type": "grounding", "name": "google_search", "content": chunks})

    if not (event.content and event.content.parts):
        return records

    for part in event.content.parts:

        if getattr(part, "text", None):
            records.append({**base, "type": "text", "name": None, "content": part.text})

        elif getattr(part, "function_call", None):
            fc = part.function_call
            records.append({
                **base, "type": "tool_call", "name": fc.name,
                "content": dict(fc.args or {}),
            })

        elif getattr(part, "function_response", None):
            fr   = part.function_response
            resp = fr.response
            # MCP fetch returns full page HTML — truncate so traces stay readable
            if isinstance(resp, dict):
                resp = {
                    k: (v[:1200] + " …[truncated]" if isinstance(v, str) and len(v) > 1200 else v)
                    for k, v in resp.items()
                }
            records.append({
                **base, "type": "tool_response", "name": fr.name,
                "content": resp,
            })

    return records


class RunTracer:
    """Collects ADK events for one candidate run and writes a JSONL trace file."""

    def __init__(self, candidate_label: str):
        ts_str    = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe      = candidate_label.replace(" ", "_").replace("/", "-")
        self.path = TRACE_DIR / f"{ts_str}_{safe}.jsonl"
        self._records: list[dict] = []
        self.grounding: list[dict] = []
        self.label = candidate_label

    def record(self, event) -> None:
        recs = _event_to_records(event, self.label)
        self._records.extend(recs)
        for r in recs:
            if r["type"] == "grounding":
                self.grounding.extend(r["content"])

    def flush(self) -> Path:
        with self.path.open("w") as f:
            for r in self._records:
                f.write(json.dumps(r, default=str) + "\n")
        return self.path

    @property
    def n_tool_calls(self) -> int:
        return sum(1 for r in self._records if r["type"] == "tool_call")

    @property
    def n_tool_responses(self) -> int:
        return sum(1 for r in self._records if r["type"] == "tool_response")

    def summary(self) -> str:
        calls = [r["name"] for r in self._records if r["type"] == "tool_call"]
        return f"{len(self._records)} events | tool calls: {calls}"


print(f"Tracer ready. Traces will be written to: {TRACE_DIR.resolve()}")

Tracer ready. Traces will be written to: /Users/connie/dev/AI-agents-google-kaggle/capstone/traces


In [255]:
# Cell 2b — URL utilities (classify_url + resolve_vertex_link)
import base64
import urllib.request
import urllib.error
from urllib.parse import urlparse, parse_qs
import re

_SEARCH_PARAM_RE = re.compile(
    r'[?&](q|query|search|searchTerm|keyword|keywords|term|terms)=',
    re.IGNORECASE,
)


def classify_url(url: str | None) -> str:
    """
    Classify a URL returned by the agent.

    Returns:
        'retailer_link' — direct product page (path present, no search params).
        'vertex_link'   — Google grounding redirect (destination unknown until followed).
        'hallucinated'  — root domain only, or a search/category page.
        'missing'       — null or empty.
    """
    if not url:
        return "missing"
    if "vertexaisearch.cloud.google.com/grounding-api-redirect/" in url:
        return "vertex_link"
    parsed = urlparse(url)
    path = parsed.path.rstrip("/")
    if not path:
        return "hallucinated"
    if _SEARCH_PARAM_RE.search(url):
        return "hallucinated"
    return "retailer_link"


def _recover_blocked_url(url: str) -> str:
    """
    Recover the real product URL from a retailer anti-bot wall.

    Some retailers (Walmart) redirect automated clients to a bot wall such as
    ``https://www.walmart.com/blocked?url=<base64 of the real path>&uuid=...``.
    The genuine product path is base64-encoded in the ``url`` query param, so we
    can reconstruct a durable product link even though the page itself refused to
    load. Non-blocked URLs are returned unchanged.
    """
    parsed = urlparse(url)
    if parsed.path.rstrip("/") != "/blocked":
        return url
    enc = (parse_qs(parsed.query).get("url") or [None])[0]
    if not enc:
        return url
    pad = enc + "=" * (-len(enc) % 4)
    for decoder in (base64.urlsafe_b64decode, base64.b64decode):
        try:
            path = decoder(pad).decode("utf-8", "replace")
        except Exception:
            continue
        if path.startswith("/"):
            return f"{parsed.scheme}://{parsed.netloc}{path}"
    return url


def resolve_vertex_link(url: str, timeout: int = 5) -> str | None:
    """Follow a Vertex grounding redirect and return the final destination URL."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"},
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return _recover_blocked_url(resp.url)
    except urllib.error.HTTPError as e:
        print(f"    vertex redirect HTTP {e.code}: {e.reason}")
        return None
    except Exception as e:
        print(f"    vertex redirect error: {type(e).__name__}: {e}")
        return None


print("URL utilities ready — classify_url(), resolve_vertex_link(), _recover_blocked_url()")


URL utilities ready — classify_url(), resolve_vertex_link(), _recover_blocked_url()


In [256]:
# Cell 5 — orchestration (price_one_candidate + run_tie_set)
from google.genai.errors import ClientError
from urllib.parse import urlparse

_session_tracers: list[RunTracer] = []

# Retailers excluded from results entirely — their listings are unreliable for a
# specific shade:
#   target.com  — search pages return unrelated products when the shade isn't listed.
#   walmart     — marketplace listings link out to third-party sellers and very
#                 often claim stock for products the store does not actually carry,
#                 producing confident false positives. Excluded across all Walmart
#                 properties (walmart.com and business.walmart.com).
# Matched as substrings of the retailer name, so "walmart" also catches
# "Walmart Marketplace", "business.walmart.com", etc.
_EXCLUDED_RETAILERS = {"target.com", "target", "ulta beauty at target", "walmart"}

# Excluded by URL host too, for entries whose retailer name doesn't say "walmart"
# but whose link points at a Walmart property. business.walmart.com (B2B) is a
# different catalog/pricing tier; walmart.com is excluded for the reason above.
_EXCLUDED_DOMAINS = {"walmart.com", "business.walmart.com"}


def _is_excluded(p: dict) -> bool:
    """True if this price entry belongs to an excluded retailer (by name or URL host)."""
    retailer = (p.get("retailer") or "").lower()
    if any(excl in retailer for excl in _EXCLUDED_RETAILERS):
        return True
    host = urlparse(p.get("url") or "").netloc.lower().removeprefix("www.")
    return host in _EXCLUDED_DOMAINS or retailer.removeprefix("www.") in _EXCLUDED_DOMAINS

_FETCH_TIMEOUT_S = 45  # max seconds to wait for one MCP fetch call


async def _run_agent(agent, message: str, tracer: RunTracer) -> str:
    """Run one agent turn, record events to tracer, return full text response."""
    runner  = InMemoryRunner(agent=agent, app_name="price_finder")
    session = await runner.session_service.create_session(
        app_name="price_finder", user_id="u1"
    )
    text = ""
    async for event in runner.run_async(
        user_id="u1",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=message)]),
    ):
        tracer.record(event)
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    text += part.text
    return text


def _parse_list(text: str) -> list[dict]:
    clean = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    try:
        parsed = json.loads(clean)
        if isinstance(parsed, list):
            return [p for p in parsed if isinstance(p, dict)]
    except (json.JSONDecodeError, ValueError):
        if text.strip():
            print(f"    JSON parse failed: {text[:200]!r}")
    return []


def _parse_obj(text: str) -> dict:
    clean = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    try:
        parsed = json.loads(clean)
        if isinstance(parsed, dict):
            return parsed
    except (json.JSONDecodeError, ValueError):
        pass
    return {}


def _backfill_urls_from_grounding(prices: list[dict], grounding: list[dict]) -> int:
    """
    Recover product URLs the model dropped, using google_search grounding chunks.

    google_search grounds every result with an authoritative (domain, title, uri)
    triple, but the model sometimes returns a retailer with ``url: null`` even
    though it clearly had the link (e.g. "ulta.com" with no URL). For each such
    entry we match a grounding chunk by domain and fill in its uri, and keep the
    chunk title as a durable identity anchor for links that may later expire.

    Returns the number of entries backfilled. Mutates `prices` in place.
    """
    if not grounding:
        return 0
    by_domain: dict[str, dict] = {}
    for ch in grounding:
        dom = (ch.get("domain") or "").lower().removeprefix("www.")
        if dom and dom not in by_domain:
            by_domain[dom] = ch

    n = 0
    for entry in prices:
        if entry.get("url") or entry.get("status"):
            continue
        retailer = (entry.get("retailer") or "").lower().removeprefix("www.").strip()
        if not retailer:
            continue
        ch = by_domain.get(retailer)
        if not ch:  # loose match on brand key: "ulta.com" ~ "Ulta Beauty" ~ "ulta"
            rkey = re.split(r"[.\s]+", retailer, maxsplit=1)[0]
            ch = next((c for d, c in by_domain.items() if d.split(".")[0] == rkey), None)
        if ch and ch.get("uri"):
            entry["url"] = ch["uri"]
            if ch.get("title") and not entry.get("product_title"):
                entry["product_title"] = ch["title"]
            n += 1
    return n


def _resolve_conflicts(prices: list[dict]) -> tuple[list[dict], list[dict]]:
    """
    Replace _clean_prices. Groups entries by canonical entity (domain from URL,
    falling back to retailer name), resolves conflicts, returns (kept, dropped).

    Resolution order within a conflict group:
      0. Verified products        — fetch-confirmed pages (brand+shade matched).
                                    Distinct URL paths/SKUs are different versions of
                                    the shade -> keep each; unverified snippets in the
                                    same domain are superseded.
      1. Exact-duplicate fast path — same price -> keep first.
      2. URL quality tiebreaker   — retailer_link > vertex_link > hallucinated.
      3. Confidence tiebreaker    — high > medium > low.
      4. Quarantine               — top entries still tied with different prices
                                    -> drop all in group.

    Each dropped entry gets a '_dropped_reason' field:
      'lower_quality' — beaten by a higher-quality entry (or a same-SKU duplicate).
      'superseded'    — unverified snippet dropped in favor of a fetch-verified page.
      'quarantined'   — unresolvable tie; conflicting prices at equal quality.
    """
    sentinels = [p for p in prices if p.get("status")]
    if sentinels:
        return sentinels, []

    prices = [p for p in prices if not _is_excluded(p)]

    def _brand_key(netloc: str) -> str:
        parts = netloc.split(".")
        return parts[-2] if len(parts) >= 2 else parts[0]

    def _fuzzy_match(a: str, b: str) -> bool:
        """True if one name is a word-boundary prefix of the other (min 3 chars)."""
        if a == b:
            return True
        short, long = (a, b) if len(a) <= len(b) else (b, a)
        if len(short) < 3:
            return False
        return long.startswith(short) and (len(long) == len(short) or long[len(short)] == " ")

    groups: dict[str, list[dict]] = {}
    name_entries: list[dict] = []

    for p in prices:
        url = p.get("url") or ""
        if url and "vertexaisearch" not in url:
            netloc = urlparse(url).netloc.lower().removeprefix("www.")
            if netloc:
                groups.setdefault(_brand_key(netloc), []).append(p)
                continue
        name_entries.append(p)

    for p in name_entries:
        retailer = (p.get("retailer") or "unknown").lower().strip()
        if re.match(r"^[\w-]+\.\w+$", retailer):
            retailer = retailer.split(".")[0]
        matched = next((b for b in groups if _fuzzy_match(retailer, b)), None)
        groups.setdefault(matched or retailer, []).append(p)

    _URL_Q = {"retailer_link": 2, "vertex_link": 1, "hallucinated": 0, "missing": -1}
    _CONF  = {"high": 2, "medium": 1, "low": 0}

    kept: list[dict] = []
    dropped: list[dict] = []

    for group in groups.values():
        if len(group) == 1:
            kept.append(group[0])
            continue

        # Fetch-confirmed real product pages take priority. Different product pages
        # (distinct URL path / SKU) are different VERSIONS of the same shade — e.g.
        # MAC Ruby Woo full-size vs travel mini — so keep each with its product info
        # instead of quarantining them as a price conflict. Same-SKU duplicates are
        # collapsed, and unverified snippets for the domain are superseded.
        verified = [p for p in group if p.get("_verified")]
        if verified:
            by_sku: dict[str, list[dict]] = {}
            for p in verified:
                sku = urlparse(p.get("url") or "").path.rstrip("/")
                by_sku.setdefault(sku, []).append(p)
            for sku_group in by_sku.values():
                best = max(sku_group, key=lambda p: (
                    _CONF.get(p.get("confidence", ""), 0), p.get("price") is not None))
                kept.append(best)
                dropped.extend({**p, "_dropped_reason": "lower_quality"}
                               for p in sku_group if p is not best)
            dropped.extend({**p, "_dropped_reason": "superseded"}
                           for p in group if not p.get("_verified"))
            continue

        price_values = {p.get("price") for p in group if p.get("price") is not None}
        if len(price_values) <= 1:
            # 0 or 1 distinct real price in this group. Keep the single best entry,
            # preferring one that actually carries the price — a null-price snippet
            # must never out-survive the entry that found the real price — then break
            # ties on URL quality and confidence.
            def _keep_score(p: dict) -> tuple[int, int, int]:
                return (
                    1 if p.get("price") is not None else 0,
                    _URL_Q.get(classify_url(p.get("url")), -1),
                    _CONF.get(p.get("confidence", ""), 0),
                )
            ranked = sorted(group, key=_keep_score, reverse=True)
            kept.append(ranked[0])
            dropped.extend({**p, "_dropped_reason": "lower_quality"} for p in ranked[1:])
            continue

        def _score(p: dict) -> tuple[int, int]:
            return (_URL_Q.get(classify_url(p.get("url")), -1), _CONF.get(p.get("confidence", ""), 0))

        ranked    = sorted(group, key=_score, reverse=True)
        top_score = _score(ranked[0])
        top_tier  = [p for p in ranked if _score(p) == top_score]

        if len(top_tier) == 1:
            kept.append(top_tier[0])
            dropped.extend({**p, "_dropped_reason": "lower_quality"} for p in ranked[1:])
        else:
            top_prices = {p.get("price") for p in top_tier if p.get("price") is not None}
            if len(top_prices) <= 1:
                kept.append(top_tier[0])
                dropped.extend({**p, "_dropped_reason": "lower_quality"} for p in ranked[1:])
            else:
                dropped.extend({**p, "_dropped_reason": "quarantined"} for p in group)

    return kept, dropped


_PUNCT_RE = re.compile(r"[^a-z0-9 ]+")


def _norm(s: str | None) -> str:
    """Lowercase, strip punctuation, collapse whitespace. 'M·A·C' -> 'm a c'."""
    return re.sub(r"\s+", " ", _PUNCT_RE.sub(" ", (s or "").lower())).strip()


def verify_product(candidate: dict, fetched: dict) -> tuple[str, str]:
    """
    Confirm a fetched page is the SAME brand + shade as the candidate.

    Returns (verdict, reason):
      'high'   — brand and shade both confirmed on the page.
      'medium' — page loaded and nothing contradicts it, but not fully confirmed.
      'reject' — page is a different product (wrong brand or wrong shade).
      'low'    — no page identity available to verify at all.

    Size / format differences (e.g. travel mini vs full size) are NOT a mismatch:
    matching is on brand + shade tokens only.
    """
    title = fetched.get("title") or ""
    hay = _norm(f"{title} {fetched.get('brand') or ''} {fetched.get('shade') or ''}")
    if not hay:
        return "low", "no page identity to verify"

    brand = _norm(candidate.get("brand"))
    shade = _norm(candidate.get("shade"))
    hay_tokens = set(hay.split())

    brand_ok = bool(brand) and (brand in hay or brand.replace(" ", "") in hay.replace(" ", ""))
    shade_ok = bool(shade) and all(tok in hay_tokens for tok in shade.split())

    if brand_ok and shade_ok:
        return "high", "brand + shade confirmed on page"
    if not brand_ok and (fetched.get("brand") or title):
        return "reject", f"brand mismatch (page: {title[:50]!r})"
    if not shade_ok and fetched.get("shade"):
        return "reject", f"shade mismatch (page shade: {fetched.get('shade')!r})"
    return "medium", "partial identity match"


async def price_one_candidate(candidate: dict, country: str = "US") -> dict:
    """
    Two-stage price lookup:
      Stage 1 — search_agent (google_search): Shopping snippets → prices + grounded URLs
                Primary query anchors to known retailers; falls back to a shorter query
                if the primary returns nothing at all. Skips fallback if the agent
                returned a status sentinel ("not_found" / "discontinued").
      Stage 1a — recover URLs the model dropped from grounding metadata
                 (authoritative domain/title/uri), then resolve vertex links:
                 follow each grounding redirect to the real retailer URL while it
                 is still live, decoding retailer bot-wall URLs when hit.
      Stage 1.5 — targeted search: for entries with a retailer name but no URL,
                  fire a focused search to recover a product page URL.
      Stage 2 — fetch_agent (MCP fetch):
                - Always fetches null-price URLs (to get the price from the page)
                - Also fetches low/medium-confidence URLs (to verify snippet prices)
                  Google Shopping snippets can show a price that the product page
                  contradicts (e.g. Walmart listing unavailable at that price).
                  If the fetch returns no price + in_stock=false, the snippet price
                  is cleared so the entry won't win in optimize().
      Post    — conflict resolution: group by retailer domain, resolve by URL
                quality then confidence, quarantine unresolvable ties.
    """
    brand, product, shade = candidate["brand"], candidate["product"], candidate["shade"]
    hint   = _COUNTRY_HINTS.get(country)
    if hint is None:
        print(f"    warning: country {country!r} not in _COUNTRY_HINTS — defaulting to USD")
        hint = "buy price USD"
    label  = f"{brand}/{shade}"
    tracer = RunTracer(label)

    # Primary: retailer-anchored query to guide Shopping panel toward known stores
    search_msg = (
        f"Find current online retail prices for: {brand} {product} in shade \"{shade}\".\n"
        f"Country: {country}. Search for: {brand} {product} {shade} {hint} ulta walmart sephora cvs\n"
        f"Return all retailers and prices as a JSON array."
    )
    # Fallback: shorter query used only if primary returns zero entries
    fallback_msg = (
        f"Find current online retail prices for: {brand} {shade} lipstick.\n"
        f"Country: {country}. Search for: {brand} {shade} lipstick {hint}\n"
        f"Return all retailers and prices as a JSON array."
    )

    # ── Stage 1: search ───────────────────────────────────────────────────────
    raw_search = ""
    try:
        raw_search = await _run_agent(search_agent, search_msg, tracer)
    except ClientError as e:
        if e.status_code == 429:
            retry_s = 62
            try:
                details = e.response_json.get("error", {}).get("details", [])
                for d in details:
                    if d.get("@type", "").endswith("RetryInfo"):
                        retry_s = int(d.get("retryDelay", "60s").rstrip("s")) + 2
            except Exception:
                pass
            print(f"    429 — waiting {retry_s}s...")
            await asyncio.sleep(retry_s)
            try:
                raw_search = await _run_agent(search_agent, search_msg, tracer)
            except Exception as e2:
                print(f"    retry failed: {e2}")
        else:
            print(f"    search error: {e}")

    prices = _parse_list(raw_search)
    # Drop excluded retailers up front so we never spend resolve/fetch calls on them.
    prices = [p for p in prices if not _is_excluded(p)]

    # Fallback: only if truly empty. A sentinel entry ("not_found" / "discontinued")
    # means the agent searched and concluded something — don't waste tokens retrying.
    if len(prices) == 0:
        print(f"    no results — retrying with short query...")
        try:
            raw_fallback = await _run_agent(search_agent, fallback_msg, tracer)
            prices = [p for p in _parse_list(raw_fallback) if not _is_excluded(p)]
        except Exception as e:
            print(f"    fallback error: {e}")

    # ── Stage 1a: recover dropped URLs from grounding metadata ────────────────
    # The model occasionally returns a retailer with url=null even though
    # google_search grounded it. Fill those from the authoritative grounding
    # chunks before we try to resolve/fetch anything.
    n_backfilled = _backfill_urls_from_grounding(prices, getattr(tracer, "grounding", []))
    if n_backfilled:
        print(f"    recovered {n_backfilled} URL(s) from grounding metadata")

    # ── Stage 1b: resolve vertex links ───────────────────────────────────────
    for entry in prices:
        if classify_url(entry.get("url")) == "vertex_link":
            resolved = resolve_vertex_link(entry["url"])
            if resolved:
                entry["url"] = resolved
            else:
                print(f"    vertex link unresolved: {entry.get('retailer', '?')}")

    # ── Stage 1.5: targeted search for no-URL entries ─────────────────────────
    no_url = [
        p for p in prices
        if not p.get("url") and p.get("retailer") and "status" not in p
    ]
    if no_url:
        print(f"    {len(no_url)} no-URL retailer(s) — firing targeted searches...")
    for entry in no_url:
        retailer = entry["retailer"]
        targeted_msg = (
            f"Find the exact product page URL for: {brand} {product} shade \"{shade}\" at {retailer}.\n"
            f"Country: {country}. Search for: {brand} {product} {shade} {retailer} {hint}\n"
            f"Return ONLY a JSON array with one entry for {retailer}."
        )
        try:
            raw_targeted = await _run_agent(search_agent, targeted_msg, tracer)
            for tp in _parse_list(raw_targeted):
                if not tp.get("url"):
                    continue
                url = tp["url"]
                if classify_url(url) == "vertex_link":
                    url = resolve_vertex_link(url) or url
                entry["url"] = url
                if entry.get("price") is None and tp.get("price") is not None:
                    entry["price"]      = tp["price"]
                    entry["currency"]   = tp.get("currency", entry.get("currency"))
                    entry["confidence"] = tp.get("confidence", "medium")
                break
        except Exception as e:
            print(f"    targeted search error ({retailer}): {e}")

    # ── Stage 2: fetch to get missing prices + verify uncertain ones ──────────
    # Fetch when: price is null (need the price) OR confidence is low/medium
    # (snippet price may be stale or from a marketplace listing).
    # High-confidence snippet prices are trusted as-is.
    fetch_entries = [
        p for p in prices
        if p.get("url") and "status" not in p
        and classify_url(p.get("url")) != "hallucinated"
        and (p.get("price") is None or p.get("confidence") in ("low", "medium", None))
    ]
    if fetch_entries:
        n_missing = sum(1 for p in fetch_entries if p.get("price") is None)
        n_verify  = len(fetch_entries) - n_missing
        parts = []
        if n_missing:
            parts.append(f"{n_missing} null-price")
        if n_verify:
            parts.append(f"{n_verify} to verify")
        print(f"    fetching {' + '.join(parts)} URL(s)...")

    for entry in fetch_entries:
        fetch_msg = (
            f"Fetch this product page and extract the price for "
            f"{brand} {product} in shade \"{shade}\":\n{entry['url']}"
        )
        try:
            raw_fetch = await asyncio.wait_for(
                _run_agent(fetch_agent, fetch_msg, tracer),
                timeout=_FETCH_TIMEOUT_S,
            )
            fetched = _parse_obj(raw_fetch)
            verdict, reason = verify_product(candidate, fetched)
            if verdict == "reject":
                # Fetched page is a different product (wrong brand/shade). Discard its
                # price entirely so a mismatched page can never win in optimize().
                entry["price"]      = None
                entry["in_stock"]   = False
                entry["confidence"] = "low"
                entry["_verify"]    = f"rejected: {reason}"
                print(f"    {entry.get('retailer', '?')}: identity check rejected — {reason}")
            elif fetched.get("price") is not None:
                # Price confirmed on a page that matches. Confidence follows IDENTITY,
                # not merely fetch success: "high" only when brand+shade both matched.
                entry["price"]      = fetched["price"]
                entry["currency"]   = fetched.get("currency", entry.get("currency", "USD"))
                entry["in_stock"]   = fetched.get("in_stock", entry.get("in_stock"))
                entry["confidence"]    = verdict         # "high" | "medium" | "low"
                entry["_verify"]       = reason
                entry["_verified"]     = verdict == "high"
                entry["product_title"] = fetched.get("title") or entry.get("product_title")
            elif fetched.get("in_stock") is False:
                # Page confirmed unavailable — clear the snippet price so this
                # entry won't be selected by optimize()
                entry["price"]      = None
                entry["in_stock"]   = False
                entry["confidence"] = "high"
                print(f"    {entry.get('retailer', '?')}: page says unavailable — snippet price cleared")
        except asyncio.TimeoutError:
            print(f"    fetch timeout ({entry.get('retailer', '?')})")
        except Exception as e:
            print(f"    fetch error ({entry.get('retailer', '?')}): {e}")

    # ── Post-process ──────────────────────────────────────────────────────────
    prices, price_conflicts = _resolve_conflicts(prices)

    tracer.flush()
    _session_tracers.append(tracer)

    return {**candidate, "prices": prices, "price_conflicts": price_conflicts}


async def run_tie_set(
    tie_set: list[dict],
    country: str = "US",
    strategy: str = "cheapest",
) -> dict:
    """
    Price each candidate sequentially, then run optimize().

    Args:
        tie_set: List of {"brand", "product", "shade", "delta_e"} dicts.
        country: Two-letter country code (default "US").
        strategy: "cheapest" (default) or "closest" — passed to optimize().
    """
    print(f"Pricing {len(tie_set)} candidates (country={country})...")
    priced = []
    for c in tie_set:
        print(f"  → {c['brand']} / {c['shade']}")
        result = await price_one_candidate(c, country)
        found  = [p for p in result.get("prices", []) if p.get("price") is not None]
        status = next((p.get("status") for p in result.get("prices", []) if p.get("status")), None)
        if status:
            print(f"    status: {status}")
        else:
            print(f"    {len(found)} price(s): {[p.get('retailer') for p in found]}")
        priced.append(result)

    outcome = optimize(priced, strategy=strategy)
    outcome["all_candidates"] = priced
    return outcome


print("Orchestration ready — two-agent pipeline (search_agent → fetch_agent).")

Orchestration ready — two-agent pipeline (search_agent → fetch_agent).


In [257]:
# Cell 5c — verify_product() unit tests
def _test_verify_product():
    cand = {"brand": "MAC", "product": "Lipstick", "shade": "Ruby Woo"}
    cases = [
        ({"title": "MAC - Ruby Woo Mini M·A·Cximal Silky Matte Lipstick | Ulta Beauty",
          "brand": "MAC", "shade": "Ruby Woo", "price": 16.0}, "high",
         "mini size, brand+shade match → high (size is NOT a mismatch)"),
        ({"title": "MAC Macximal Silky Matte Lipstick Ruby Woo | Ulta",
          "brand": "MAC", "shade": "Ruby Woo", "price": 25.0}, "high",
         "full size, brand+shade match → high"),
        ({"title": "MAC Matte Lipstick Velvet Teddy", "brand": "MAC",
          "shade": "Velvet Teddy", "price": 24.0}, "reject",
         "same brand, WRONG shade (Velvet Teddy) → reject"),
        ({"title": "NYX Soft Matte Lip Cream Monte Carlo", "brand": "NYX",
          "shade": "Monte Carlo", "price": 8.0}, "reject",
         "WRONG brand (NYX) → reject"),
        ({"title": None, "brand": None, "shade": None, "price": 20.0}, "low",
         "price but no identity info → low (cannot verify)"),
    ]
    n_ok = 0
    for fetched, exp, label in cases:
        got, reason = verify_product(cand, fetched)
        ok = got == exp
        n_ok += ok
        print(f"  {'✓' if ok else '✗'} {label}  → {got} ({reason})")
    print(f"\n  {n_ok} / {len(cases)} passed")

_test_verify_product()


  ✓ mini size, brand+shade match → high (size is NOT a mismatch)  → high (brand + shade confirmed on page)
  ✓ full size, brand+shade match → high  → high (brand + shade confirmed on page)
  ✓ same brand, WRONG shade (Velvet Teddy) → reject  → reject (shade mismatch (page shade: 'Velvet Teddy'))
  ✓ WRONG brand (NYX) → reject  → reject (brand mismatch (page: 'NYX Soft Matte Lip Cream Monte Carlo'))
  ✓ price but no identity info → low (cannot verify)  → low (no page identity to verify)

  5 / 5 passed


In [258]:
# Cell 5b — _resolve_conflicts unit tests
def _test_resolve_conflicts():
    def _e(retailer, price, url=None, confidence="high", in_stock=True):
        return {"retailer": retailer, "price": price, "currency": "USD",
                "in_stock": in_stock, "url": url, "confidence": confidence}

    n_ok = n_fail = 0

    def check(label, kept, dropped, *, n_kept, prices_kept, reasons_dropped):
        nonlocal n_ok, n_fail
        got_prices  = sorted(p["price"] for p in kept  if p.get("price") is not None)
        got_reasons = sorted(p.get("_dropped_reason", "none") for p in dropped)
        passed = (
            len(kept) == n_kept and
            got_prices == sorted(prices_kept) and
            got_reasons == sorted(reasons_dropped)
        )
        print(f"  {'✓' if passed else '✗'} {label}")
        if not passed:
            print(f"       kept:    {[(p.get('retailer'), p.get('price')) for p in kept]}")
            print(f"       dropped: {got_reasons}  (expected {sorted(reasons_dropped)})")
        n_ok   += passed
        n_fail += not passed

    # ── Grouping: TLD-strip and fuzzy match ──────────────────────────────────────
    # These tests verify that entries ARE grouped together (n_kept=1, not 2).
    # When one has a URL and one doesn't, the URL entry has higher quality and wins
    # (lower_quality drop). Quarantine only fires when both have equal quality.
    print("Grouping: TLD-strip and fuzzy match")

    # Same domain, same price → dedup
    k, d = _resolve_conflicts([
        _e("Sephora", 18.0, "https://www.sephora.com/product/123"),
        _e("Sephora", 18.0, "https://www.sephora.com/product/456"),
    ])
    check("exact duplicate same domain → dedup", k, d,
          n_kept=1, prices_kept=[18.0], reasons_dropped=["lower_quality"])

    # TLD-strip: sephora.com URL + no-URL, different prices
    # → grouped (n_kept=1 proves it), URL entry wins as higher quality
    k, d = _resolve_conflicts([
        _e("Sephora", 18.0,  "https://www.sephora.com/product/123"),
        _e("Sephora", 21.95, None),
    ])
    check("TLD-strip: sephora.com + no-URL → grouped, URL entry wins", k, d,
          n_kept=1, prices_kept=[18.0], reasons_dropped=["lower_quality"])

    # Fuzzy: ulta.com + "Ulta Beauty" no-URL, different prices
    # → grouped (n_kept=1 proves it), URL entry wins
    k, d = _resolve_conflicts([
        _e("Ulta Beauty", 27.0, "https://www.ulta.com/ip/123"),
        _e("Ulta Beauty", 29.0, None),
    ])
    check("fuzzy: ulta.com + 'Ulta Beauty' no-URL → grouped, URL entry wins", k, d,
          n_kept=1, prices_kept=[27.0], reasons_dropped=["lower_quality"])

    # Fuzzy same price → dedup, not quarantine
    k, d = _resolve_conflicts([
        _e("Ulta Beauty", 27.0, "https://www.ulta.com/ip/123"),
        _e("Ulta Beauty", 27.0, None),
    ])
    check("fuzzy same price → dedup", k, d,
          n_kept=1, prices_kept=[27.0], reasons_dropped=["lower_quality"])

    # Fuzzy: cvs.com + "CVS Pharmacy" no-URL → grouped, URL entry wins
    k, d = _resolve_conflicts([
        _e("CVS Pharmacy", 12.0, "https://www.cvs.com/product/123"),
        _e("CVS Pharmacy", 14.0, None),
    ])
    check("fuzzy: cvs.com + 'CVS Pharmacy' no-URL → grouped, URL entry wins", k, d,
          n_kept=1, prices_kept=[12.0], reasons_dropped=["lower_quality"])

    # Without the grouping fix, two entries with different names would both be
    # kept (n_kept=2). Verify they're still separate entities when domains differ.
    k, d = _resolve_conflicts([
        _e("MAC", 22.0, "https://www.maccosmetics.com/product/123"),
        _e("Macys", 20.0, "https://www.macys.com/product/456"),
    ])
    check("no false match: maccosmetics vs macys → separate groups, both kept", k, d,
          n_kept=2, prices_kept=[22.0, 20.0], reasons_dropped=[])

    # Different retailers → all kept
    k, d = _resolve_conflicts([
        _e("Ulta Beauty", 27.0, "https://ulta.com/product/123"),
        _e("Sephora",     28.0, "https://sephora.com/product/456"),
    ])
    check("different retailers → both kept", k, d,
          n_kept=2, prices_kept=[27.0, 28.0], reasons_dropped=[])

    # ── Quarantine: equal quality, conflicting prices ────────────────────────────
    print("Quarantine: equal quality, conflicting prices")

    # Two hallucinated search-page URLs, same retailer → quarantine
    k, d = _resolve_conflicts([
        _e("Sephora", 18.0,  "https://sephora.com/search?q=mac"),
        _e("Sephora", 21.95, "https://sephora.com/search?q=lipstick"),
    ])
    check("two hallucinated URLs, different prices → quarantine", k, d,
          n_kept=0, prices_kept=[], reasons_dropped=["quarantined", "quarantined"])

    # Two no-URL entries for the same retailer → quarantine
    k, d = _resolve_conflicts([
        _e("Sephora", 18.0,  None),
        _e("Sephora", 21.95, None),
    ])
    check("two no-URL entries, different prices → quarantine", k, d,
          n_kept=0, prices_kept=[], reasons_dropped=["quarantined", "quarantined"])

    # Two retailer_link URLs, different prices → quarantine
    k, d = _resolve_conflicts([
        _e("Sephora", 28.0, "https://sephora.com/product/123"),
        _e("Sephora", 31.0, "https://sephora.com/product/456"),
    ])
    check("two retailer_link URLs, different prices → quarantine", k, d,
          n_kept=0, prices_kept=[], reasons_dropped=["quarantined", "quarantined"])

    # ── Conflict resolution scoring ──────────────────────────────────────────────
    print("Conflict resolution scoring")

    # URL quality: retailer_link beats hallucinated even when cheaper
    k, d = _resolve_conflicts([
        _e("Sephora", 18.0, "https://sephora.com/search?q=mac"),   # hallucinated
        _e("Sephora", 21.0, "https://sephora.com/product/123"),    # retailer_link
    ])
    check("URL quality: retailer_link ($21) beats hallucinated ($18)", k, d,
          n_kept=1, prices_kept=[21.0], reasons_dropped=["lower_quality"])

    # Confidence tiebreaker: high beats medium at equal URL quality
    k, d = _resolve_conflicts([
        _e("Sephora", 28.0, "https://sephora.com/product/123", confidence="medium"),
        _e("Sephora", 30.0, "https://sephora.com/product/456", confidence="high"),
    ])
    check("confidence tiebreaker: high ($30) beats medium ($28)", k, d,
          n_kept=1, prices_kept=[30.0], reasons_dropped=["lower_quality"])

    # ── Edge cases ───────────────────────────────────────────────────────────────
    print("Edge cases")

    # Excluded retailer stripped before grouping
    k, d = _resolve_conflicts([
        _e("Target", 15.0, "https://target.com/product"),
        _e("MAC",    22.0, "https://maccosmetics.com/product"),
    ])
    check("excluded retailer (Target) removed", k, d,
          n_kept=1, prices_kept=[22.0], reasons_dropped=[])

    # Walmart is excluded wholesale (consumer walmart.com too): marketplace listings
    # link to third-party sellers and often claim stock the store doesn't carry.
    k, d = _resolve_conflicts([
        _e("Walmart", 8.99, "https://www.walmart.com/ip/nyx-monte-carlo/123"),
        _e("MAC",     22.0, "https://maccosmetics.com/product"),
    ])
    check("excluded retailer (Walmart consumer) removed", k, d,
          n_kept=1, prices_kept=[22.0], reasons_dropped=[])

    # Sentinel pass-through
    k, d = _resolve_conflicts([{"status": "not_found", "retailer": None, "price": None}])
    sentinel_ok = len(k) == 1 and k[0].get("status") == "not_found" and len(d) == 0
    print(f"  {'✓' if sentinel_ok else '✗'} sentinel (not_found) passes through unchanged")
    n_ok += sentinel_ok; n_fail += not sentinel_ok

    # Single entry → always kept
    k, d = _resolve_conflicts([_e("Sephora", 28.0, "https://sephora.com/product/123")])
    check("single entry → kept", k, d,
          n_kept=1, prices_kept=[28.0], reasons_dropped=[])

    # ── Regression: reported wrong-result fixes ──────────────────────────────────
    print("Regression")

    # MAC Ruby Woo: a null-price snippet sorted first and dropped the real $16 mini.
    # Fast path must keep the entry that actually carries the price.
    k, d = _resolve_conflicts([
        _e("Ulta Beauty", None, "https://www.ulta.com/p/macximal?sku=2621407"),
        _e("Ulta Beauty", 16.0, "https://www.ulta.com/p/mini-macximal?sku=2624231"),
    ])
    check("null-price entry does not out-survive real price → keep $16", k, d,
          n_kept=1, prices_kept=[16.0], reasons_dropped=["lower_quality"])

    # NYX: business.walmart.com (B2B) excluded — belt-and-suspenders with the
    # name-based Walmart exclusion, this confirms host-based exclusion also fires.
    k, d = _resolve_conflicts([
        _e("Walmart", 8.49, "https://business.walmart.com/ip/nyx/999"),
        _e("Sephora", 12.0, "https://www.sephora.com/product/123"),
    ])
    check("business.walmart.com excluded by domain", k, d,
          n_kept=1, prices_kept=[12.0], reasons_dropped=[])

    # ── Verified products: different versions kept, snippets superseded ──────────
    print("Verified products")

    def _ve(retailer, price, url, title):
        return {"retailer": retailer, "price": price, "currency": "USD",
                "in_stock": True, "url": url, "confidence": "high",
                "_verified": True, "product_title": title}

    # Two fetch-verified MAC Ruby Woo versions at Ulta (full $25 + mini $16):
    # different SKUs → both kept with product info, NOT quarantined.
    k, d = _resolve_conflicts([
        _ve("Ulta Beauty", 25.0, "https://www.ulta.com/p/macximal?sku=2621407", "Macximal Silky Matte Lipstick"),
        _ve("Ulta Beauty", 16.0, "https://www.ulta.com/p/mini-macximal?sku=2624231", "Mini Macximal Silky Matte Lipstick"),
    ])
    check("two verified Ulta versions → both kept (no quarantine)", k, d,
          n_kept=2, prices_kept=[16.0, 25.0], reasons_dropped=[])

    # Verified page + unverified no-URL snippet at same domain → snippet superseded.
    k, d = _resolve_conflicts([
        _ve("Ulta Beauty", 25.0, "https://www.ulta.com/p/macximal?sku=2621407", "Macximal Silky Matte Lipstick"),
        _e("Ulta Beauty", 19.0, None),
    ])
    check("verified page supersedes unverified snippet", k, d,
          n_kept=1, prices_kept=[25.0], reasons_dropped=["superseded"])

    # Same verified SKU returned twice → collapse to one.
    k, d = _resolve_conflicts([
        _ve("Ulta Beauty", 25.0, "https://www.ulta.com/p/macximal?sku=2621407", "Macximal Silky Matte Lipstick"),
        _ve("Ulta Beauty", 25.0, "https://www.ulta.com/p/macximal?sku=2621407", "Macximal Silky Matte Lipstick"),
    ])
    check("same verified SKU twice → dedup", k, d,
          n_kept=1, prices_kept=[25.0], reasons_dropped=["lower_quality"])

    print(f"\n  {n_ok} / {n_ok + n_fail} passed")

_test_resolve_conflicts()

Grouping: TLD-strip and fuzzy match
  ✓ exact duplicate same domain → dedup
  ✓ TLD-strip: sephora.com + no-URL → grouped, URL entry wins
  ✓ fuzzy: ulta.com + 'Ulta Beauty' no-URL → grouped, URL entry wins
  ✓ fuzzy same price → dedup
  ✓ fuzzy: cvs.com + 'CVS Pharmacy' no-URL → grouped, URL entry wins
  ✓ no false match: maccosmetics vs macys → separate groups, both kept
  ✓ different retailers → both kept
Quarantine: equal quality, conflicting prices
  ✓ two hallucinated URLs, different prices → quarantine
  ✓ two no-URL entries, different prices → quarantine
  ✓ two retailer_link URLs, different prices → quarantine
Conflict resolution scoring
  ✓ URL quality: retailer_link ($21) beats hallucinated ($18)
  ✓ confidence tiebreaker: high ($30) beats medium ($28)
Edge cases
  ✓ excluded retailer (Target) removed
  ✓ excluded retailer (Walmart consumer) removed
  ✓ sentinel (not_found) passes through unchanged
  ✓ single entry → kept
Regression
  ✓ null-price entry does not out-survive 

## 6. Demo run

Two candidates chosen to exercise both pipeline stages:
- **NYX Monte Carlo** — frequently returns null-price Vertex redirect URLs, which triggers `fetch_agent` (MCP Stage 2)
- **MAC Ruby Woo** — reliably returns snippet prices, guarantees a recommendation

After both are priced, `optimize()` picks the cheapest in-stock option and flags a tradeoff
if the closest color match costs meaningfully more.

In [259]:
# Cell 6 — demo run (NYX + MAC; swap to TIE_SET for all 4)
result = await run_tie_set([TIE_SET[0], TIE_SET[2]], country="US")

rec = result["recommendation"]
if rec:
    r = result["retailer"]
    print(f"\n✓ Recommendation: {rec['brand']} — {rec['product']} in {rec['shade']}")
    print(f"  Best price: {r.get('currency', '')} {r.get('price', '?')} at {r.get('retailer', '')}")
    print(f"  In stock:   {r.get('in_stock')}")
    if result["tradeoff"] and result["alternative"]:
        alt   = result["alternative"]
        alt_r = alt["_best_retailer"]
        print(f"\n  ⚠️  Tradeoff: closest color is {alt['brand']} {alt['shade']}")
        print(f"     at {alt_r.get('currency', '')} {alt_r.get('price', '?')} — verify at checkout")
else:
    print("No in-stock candidates found.", result.get("note", ""))

Pricing 2 candidates (country=US)...
  → NYX / Monte Carlo
    2 no-URL retailer(s) — firing targeted searches...
    0 price(s): []
  → MAC / Ruby Woo
    fetching 2 null-price URL(s)...
    2 price(s): ['Ulta Beauty', 'Ulta Beauty']

✓ Recommendation: MAC — Lipstick in Ruby Woo
  Best price: USD 16.0 at Ulta Beauty
  In stock:   True


## 7. Results renderer

`render_results(result)` takes the dict returned by `run_tie_set` and displays HTML
cards — one for the recommended pick, and (if a tradeoff was flagged) one for the
closest-color alternative.

Each card shows:
- Brand, product line, shade name, and ΔE from the target color
- Best price found and the retailer it came from
- In-stock status (green / red / grey if unknown)
- Confidence level of the price extraction (`high` / `medium` / `low`)
- A direct link to the product listing

Prices are extracted from live pages at run time and shown as estimates —
the card includes a "verify at checkout" reminder because prices can change
between when the agent reads the page and when the user clicks through.

In [260]:
# Cell 7 — render_results() (best pick card + all prices table)
import html
from IPython.display import display, HTML


h = html.escape  # escape all LLM-derived strings before HTML interpolation


def render_results(result: dict) -> None:
    """Display price-finder results: recommendation card(s) + all prices found per candidate."""
    rec        = result.get("recommendation")
    alt        = result.get("alternative")
    tradeoff   = result.get("tradeoff", False)
    candidates = result.get("all_candidates", [])
    rec_shade  = rec.get("shade") if rec else None

    def _card(candidate: dict | None, label: str, badge_color: str) -> str:
        if not candidate:
            return ""
        retailer = candidate.get("_best_retailer") or {}
        price    = retailer.get("price")
        currency = retailer.get("currency", "")
        in_stock = retailer.get("in_stock")
        url      = retailer.get("url", "")
        conf     = retailer.get("confidence", "")
        delta_e  = candidate.get("delta_e", "?")

        price_str = f"{h(currency)} {price:.2f}" if price is not None else "price not found"
        stock_str = "In stock" if in_stock else ("Out of stock" if in_stock is False else "Unknown")
        stock_col = "#22c55e" if in_stock else ("#ef4444" if in_stock is False else "#94a3b8")
        conf_col  = {"high": "#22c55e", "medium": "#f59e0b", "low": "#ef4444"}.get(conf, "#94a3b8")
        link      = f'<a href="{h(url)}" target="_blank">View listing →</a>' if url else ""

        return f"""
        <div style="border:1px solid #e2e8f0;border-radius:8px;padding:16px;margin-bottom:12px">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:8px">
            <span style="background:{badge_color};color:#fff;font-size:11px;
                         font-weight:bold;padding:3px 10px;border-radius:4px">{label}</span>
            <b>{h(candidate.get('brand', ''))}</b> — {h(candidate.get('product', ''))}
          </div>
          <div style="color:#6366f1;margin-bottom:6px">
            Shade: {h(candidate.get('shade', ''))} &nbsp;·&nbsp; ΔE = {h(str(delta_e))}
          </div>
          <div style="font-size:22px;font-weight:bold;margin-bottom:4px">{price_str}</div>
          <div style="margin-bottom:4px">
            <span style="color:#64748b">Retailer: {h(retailer.get('retailer', ''))}</span>
            &nbsp;·&nbsp;
            <span style="color:{stock_col};font-weight:bold">{stock_str}</span>
          </div>
          <div style="margin-bottom:8px">
            <small>Confidence: <span style="color:{conf_col};font-weight:bold">{conf or 'n/a'}</span>
            &nbsp;·&nbsp;
            <span style="color:#94a3b8">Prices are estimates — verify at checkout</span></small>
          </div>
          {link}
        </div>"""

    parts = []

    # ── Recommendation card(s) ────────────────────────────────────────────────
    if rec:
        parts.append(_card(rec, "BEST PICK", "#6366f1"))
        if tradeoff and alt:
            parts.append(
                "<div style='color:#f59e0b;font-weight:bold;margin:8px 0'>"
                "⚠️ Tradeoff: the closest color match costs more — shown below for comparison"
                "</div>"
            )
            parts.append(_card(alt, "CLOSEST COLOR", "#f59e0b"))
    else:
        parts.append("<p style='color:#ef4444'>No in-stock candidates found.</p>")

    # ── All prices found ──────────────────────────────────────────────────────
    if candidates:
        parts.append("<h4 style='margin:24px 0 12px;color:#1e293b'>All prices found</h4>")
        for c in candidates:
            is_rec       = c.get("shade") == rec_shade
            border_color = "#6366f1" if is_rec else "#e2e8f0"
            prices       = c.get("prices", [])
            status       = next((p.get("status") for p in prices if p.get("status")), None)

            best_badge = (
                "&nbsp;<span style='background:#6366f1;color:#fff;font-size:10px;"
                "padding:1px 6px;border-radius:3px'>BEST PICK</span>"
                if is_rec else ""
            )
            header = (
                f'<div style="border-left:3px solid {border_color};padding-left:12px;margin-bottom:16px">'
                f'<div style="font-weight:bold;margin-bottom:6px">'
                f'{h(c["brand"])} — {h(c["product"])} '
                f'<span style="color:#6366f1">{h(c["shade"])}</span>'
                f'&nbsp;<span style="color:#94a3b8;font-size:11px">ΔE = {c.get("delta_e","?")}</span>'
                f'{best_badge}'
                f'</div>'
            )

            if status:
                body = f'<p style="color:#94a3b8;font-size:12px;margin:4px 0">status: <b>{h(status)}</b></p>'
            elif not prices:
                body = '<p style="color:#94a3b8;font-size:12px;margin:4px 0">no results</p>'
            else:
                rows = []
                for p in prices:
                    if p.get("status"):
                        continue
                    price     = p.get("price")
                    price_str = f'{h(p.get("currency", ""))} {price:.2f}' if price is not None else "—"
                    in_stock  = p.get("in_stock")
                    stock_str = "✓" if in_stock else ("✗" if in_stock is False else "?")
                    stock_col = "#22c55e" if in_stock else ("#ef4444" if in_stock is False else "#94a3b8")
                    conf      = p.get("confidence", "")
                    conf_col  = {"high": "#22c55e", "medium": "#f59e0b", "low": "#ef4444"}.get(conf, "#94a3b8")
                    url       = p.get("url")
                    retailer  = p.get("retailer") or "unknown"
                    ret_cell  = (
                        f'<a href="{h(url)}" target="_blank" style="color:#3b82f6">{h(retailer)}</a>'
                        if url else h(retailer)
                    )
                    variant = p.get("product_title")
                    if variant:
                        ret_cell += (
                            f'<div style="font-size:10px;color:#94a3b8;max-width:280px">'
                            f'{h(variant)}</div>'
                        )
                    rows.append(
                        f'<tr>'
                        f'<td style="padding:3px 16px 3px 0;font-size:12px">{ret_cell}</td>'
                        f'<td style="padding:3px 16px 3px 0;font-size:12px;font-weight:bold">{price_str}</td>'
                        f'<td style="padding:3px 16px 3px 0;font-size:12px;color:{stock_col}">{stock_str}</td>'
                        f'<td style="padding:3px 0;font-size:11px;color:{conf_col}">{conf}</td>'
                        f'</tr>'
                    )
                body = (
                    '<table style="border-collapse:collapse">'
                    '<thead><tr>'
                    + "".join(
                        f'<th style="padding:2px 16px 4px 0;font-size:11px;color:#94a3b8;'
                        f'font-weight:normal;text-align:left">{col}</th>'
                        for col in ["Retailer", "Price", "In stock", "Confidence"]
                    )
                    + f'</tr></thead><tbody>{"".join(rows)}</tbody></table>'
                )

            conflicts     = c.get("price_conflicts", [])
            n_quarantined = sum(1 for p in conflicts if p.get("_dropped_reason") == "quarantined")
            n_lower       = sum(1 for p in conflicts if p.get("_dropped_reason") == "lower_quality")
            n_superseded  = sum(1 for p in conflicts if p.get("_dropped_reason") == "superseded")
            if conflicts:
                label_parts = []
                if n_quarantined:
                    label_parts.append(f"{n_quarantined} quarantined")
                if n_superseded:
                    label_parts.append(f"{n_superseded} superseded")
                if n_lower:
                    label_parts.append(f"{n_lower} lower quality")
                conflict_rows = []
                for cp in conflicts:
                    reason   = cp.get("_dropped_reason", "")
                    rc       = "#ef4444" if reason == "quarantined" else "#94a3b8"
                    cp_price = cp.get("price")
                    cp_pstr  = f'{h(cp.get("currency", ""))} {cp_price:.2f}' if cp_price is not None else "—"
                    cp_url   = cp.get("url")
                    cp_ret   = cp.get("retailer") or "unknown"
                    cp_cell  = (
                        f'<a href="{h(cp_url)}" target="_blank" style="color:#3b82f6">{h(cp_ret)}</a>'
                        if cp_url else h(cp_ret)
                    )
                    conflict_rows.append(
                        f'<tr>'
                        f'<td style="padding:3px 16px 3px 0;font-size:12px">{cp_cell}</td>'
                        f'<td style="padding:3px 16px 3px 0;font-size:12px">{cp_pstr}</td>'
                        f'<td style="padding:3px 0;font-size:11px;color:{rc}">{h(reason)}</td>'
                        f'</tr>'
                    )
                conflict_html = (
                    f'<details style="margin-top:6px">'
                    f'<summary style="color:#94a3b8;font-size:11px;cursor:pointer">▸ {len(conflicts)} dropped ({", ".join(label_parts)})</summary>'
                    '<table style="border-collapse:collapse;margin-top:4px"><thead><tr>'
                    + "".join(
                        f'<th style="padding:2px 16px 4px 0;font-size:11px;color:#94a3b8;font-weight:normal;text-align:left">{col}</th>'
                        for col in ["Retailer", "Price", "Reason"]
                    )
                    + f'</tr></thead><tbody>{"".join(conflict_rows)}</tbody></table></details>'
                )
            else:
                conflict_html = ""
            parts.append(header + body + conflict_html + "</div>")

    display(HTML("\n".join(parts)))


render_results(result)

Retailer,Price,In stock,Confidence
Ulta Beauty,—,✓,none
CVS,—,✓,none
Retailer,Price,In stock,Confidence
Ulta BeautyMAC - Ruby Woo M·A·Cximal Silky Matte Lipstick | Ulta Beauty,USD 25.00,?,high
Ulta BeautyMAC - Ruby Woo Mini M·A·Cximal Silky Matte Lipstick | Ulta Beauty,USD 16.00,✓,high


## 8. Trace viewer

`show_trace(path)` renders a JSONL trace file as a color-coded HTML event timeline,
making it easy to see exactly what the agent did for one URL:

- **Blue rows** — `tool_call`: agent called `fetch(url=...)` with these arguments
- **Green rows** — `tool_response`: what `mcp-server-fetch` returned (HTML, truncated)
- **Purple rows** — `text`: agent's own text output (the final JSON price record)

`analyze_traces()` prints a session-level summary across all candidates that have run
in this kernel session: total event counts, tool call distribution, all URLs fetched,
and how many text turns contained a price. This is useful for diagnosing patterns
like "agent fetched the page but returned null price" (look at the tool_response row
to see what the HTML actually contained) or "no fetch calls at all" (agent may have
errored before calling the tool).

The two convenience cells below (`_session_tracers[-1]` and `analyze_traces()`) can be
run at any point after the demo cell to inspect what happened.

In [261]:
# Cell 8 — trace viewer (show_trace + analyze_traces)
from IPython.display import display, HTML
from collections import Counter


_COLORS = {
    "tool_call":     ("#dbeafe", "#1d4ed8"),
    "tool_response": ("#dcfce7", "#15803d"),
    "text":          ("#f3e8ff", "#7e22ce"),
}
_DEFAULT_COLOR = ("#f1f5f9", "#334155")


def show_trace(path, max_content_len: int = 400) -> None:
    path = Path(path)
    if not path.exists():
        print(f"File not found: {path}")
        return

    records = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    if not records:
        print("Empty trace file.")
        return

    candidate = records[0].get("candidate", path.stem)
    rows = []
    for i, r in enumerate(records):
        bg, fg = _COLORS.get(r["type"], _DEFAULT_COLOR)
        content = r.get("content", "")
        if isinstance(content, dict):
            content_str = json.dumps(content, indent=2)
        else:
            content_str = str(content)
        if len(content_str) > max_content_len:
            content_str = content_str[:max_content_len] + " …"

        name_tag = (
            f'<span style="font-family:monospace;font-size:11px;'
            f'background:#fff4;padding:1px 6px;border-radius:3px">'
            f'{r.get("name") or ""}</span> '
            if r.get("name") else ""
        )

        rows.append(f"""
        <tr style="background:{bg};color:{fg};vertical-align:top">
          <td style="padding:4px 8px;font-size:11px;white-space:nowrap;color:#64748b">{i+1}</td>
          <td style="padding:4px 8px;font-size:11px;white-space:nowrap"><b>{r['type']}</b></td>
          <td style="padding:4px 8px;font-size:11px;white-space:nowrap">{name_tag}</td>
          <td style="padding:4px 8px;font-size:11px;font-family:monospace;
                     white-space:pre-wrap;max-width:600px;word-break:break-all">{content_str}</td>
        </tr>""")

    display(HTML(f"""
    <h4 style="margin-bottom:6px">Trace: {candidate}
      <small style="color:#64748b;font-weight:normal">({len(records)} events)</small>
    </h4>
    <table style="border-collapse:collapse;width:100%;font-size:12px">
      <thead>
        <tr style="background:#f8fafc;color:#475569;font-size:11px">
          <th style="padding:4px 8px;text-align:left">#</th>
          <th style="padding:4px 8px;text-align:left">type</th>
          <th style="padding:4px 8px;text-align:left">tool</th>
          <th style="padding:4px 8px;text-align:left">content</th>
        </tr>
      </thead>
      <tbody>{"".join(rows)}</tbody>
    </table>
    """))


def analyze_traces(tracers: list | None = None) -> None:
    if tracers is None:
        tracers = _session_tracers
    if not tracers:
        print("No traces yet — run at least one candidate first.")
        return

    all_records = []
    for t in tracers:
        path = t.path
        if path.exists():
            all_records.extend(json.loads(l) for l in path.read_text().splitlines() if l.strip())

    if not all_records:
        print("Trace files are empty.")
        return

    tool_calls     = [r for r in all_records if r["type"] == "tool_call"]
    tool_responses = [r for r in all_records if r["type"] == "tool_response"]
    texts          = [r for r in all_records if r["type"] == "text"]

    print(f"── Session trace summary ({'  |  '.join(t.label for t in tracers)}) ──")
    print(f"  Total events:    {len(all_records)}")
    print(f"  Tool calls:      {len(tool_calls)}")
    print(f"  Tool responses:  {len(tool_responses)}")
    print(f"  Text turns:      {len(texts)}")
    print()

    call_counts = Counter(r["name"] for r in tool_calls)
    print("  Tool call distribution:")
    for name, n in call_counts.most_common():
        print(f"    {name}: {n}×")
    print()

    search_calls = [r for r in tool_calls if r["name"] == "google_search"]
    if search_calls:
        print(f"  Google searches ({len(search_calls)}):")
        for r in search_calls:
            c = r.get("content", {})
            query = c.get("query") or c.get("q") or str(c)
            print(f"    [{r.get('candidate','?')}] {query}")
        print()

    prices_found = sum(1 for r in texts if "$" in str(r.get("content","")) or
                       "price" in str(r.get("content","")).lower())
    print(f"  Text turns mentioning prices: {prices_found} / {len(texts)}")


print("Trace viewer ready. Call show_trace(path) or analyze_traces().")

Trace viewer ready. Call show_trace(path) or analyze_traces().


In [262]:
# Cell 8b — show most recent trace
if _session_tracers:
    show_trace(_session_tracers[-1].path)
else:
    print("No traces yet — run a candidate first (Cell 6 or any V cell).")

#,type,tool,content
1,text,,"[ { ""retailer"": ""Ulta Beauty"", ""price"": null, ""currency"": ""USD"", ""in_stock"": true, ""url"": ""https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEE-KqzcpfDewOBv-Kqo3K3oVYB0yBmc-VyxgBaNzpzD5AYjdJpWDXmicJcfnqPxVNHz4gaydRARtRjpc7eaJ2hig5cvUiDgQshGek2FmgC2vqPPWoo9ylvy7LwZu7iFvisWEzoTHmXGAh4LUHj-prsv61Eyso2poFVd1hMXLXhtwS_k7wlPDSelhAy-_e4Z4DRyKkp5ruoGf90dHIflNqmuGMc …"
2,tool_call,fetch,"{ ""url"": ""https://www.ulta.com/p/macximal-silky-matte-lipstick-pimprod2043558?sku=2621407&_dl=285123%3B237235954&sharedID=285123&pr_rd_page=2"", ""max_length"": 20000 }"
3,tool_response,fetch,"{ ""content"": [ { ""type"": ""text"", ""text"": ""Contents of https://www.ulta.com/p/macximal-silky-matte-lipstick-pimprod2043558?sku=2621407&_dl=285123%3B237235954&sharedID=285123&pr_rd_page=2:\nMAC - Ruby Woo M\u00b7A\u00b7Cximal Silky Matte Lipstick | Ulta Beauty\n\nSKIP TO MAIN\n\nSKIP TO FOOTER\n\nBeauty in 2 hours or less\u2014order online, pick up in store.\n\nUse the following zo …"
4,text,,"```json { ""title"": ""MAC - Ruby Woo M·A·Cximal Silky Matte Lipstick | Ulta Beauty"", ""brand"": ""MAC"", ""shade"": ""Ruby Woo"", ""price"": 25.00, ""currency"": ""USD"", ""in_stock"": null } ```"
5,tool_call,fetch,"{ ""max_length"": 20000, ""url"": ""https://www.ulta.com/p/mini-macximal-silky-matte-lipstick-pimprod2045123?sku=2624231"" }"
6,tool_response,fetch,"{ ""content"": [ { ""type"": ""text"", ""text"": ""Contents of https://www.ulta.com/p/mini-macximal-silky-matte-lipstick-pimprod2045123?sku=2624231:\nMAC - Ruby Woo Mini M\u00b7A\u00b7Cximal Silky Matte Lipstick | Ulta Beauty\n\nSKIP TO MAIN\n\nSKIP TO FOOTER\n\nBeauty in 2 hours or less\u2014order online, pick up in store.\n\nUse the following zoom and pan buttons to control the image th …"
7,text,,"```json {""title"": ""MAC - Ruby Woo Mini M·A·Cximal Silky Matte Lipstick | Ulta Beauty"", ""brand"": ""MAC"", ""shade"": ""Ruby Woo"", ""price"": 16.00, ""currency"": ""USD"", ""in_stock"": true} ```"


In [209]:
# Cell 8c — cross-candidate trace summary
analyze_traces()

── Session trace summary (NYX/Monte Carlo  |  MAC/Ruby Woo) ──
  Total events:    121
  Tool calls:      57
  Tool responses:  56
  Text turns:      8

  Tool call distribution:
    fetch: 57×

  Text turns mentioning prices: 8 / 8


## 9. Verification checklist

Five cells below satisfy the verification steps from the brief. Run them in order after
the demo cell, or independently if you want to test one scenario at a time.

| Cell | Verifies | Expected outcome |
|---|---|---|
| V1 | Single product, US | Real price + working retailer URL for MAC Ruby Woo |
| V2 | Full tie set (4 candidates) | All candidates priced, `optimize()` returns cheapest in-stock |
| V3 | Forced tradeoff | `tradeoff=True`, both cheapest and closest shown |
| V4 | Country change → UK | UK retailers surfaced, currency = GBP |
| V5 | No secrets in code | Grep confirms no hardcoded API keys |

In [ ]:
# Cell V1 — single product (MAC Ruby Woo)
single = await run_tie_set([TIE_SET[2]], country="US")
rec = single["recommendation"]
print("V1 — single product price:")
if rec:
    r = single["retailer"]
    print(f"  {rec['brand']} {rec['shade']}: {r.get('currency')} {r.get('price')} @ {r.get('retailer')}")
    print(f"  URL: {r.get('url', 'none')}")
else:
    print("  No result found — check agent debug output above")

In [ ]:
# Cell V2 — full tie set (all 4 candidates)
full_result = await run_tie_set(TIE_SET, country="US")
print("\nV2 — full tie set:")
for c in full_result["all_candidates"]:
    n = sum(1 for p in c.get("prices", []) if p.get("price") is not None)
    print(f"  {c['brand']} {c['shade']}: {n} price(s)")
rec = full_result["recommendation"]
if rec:
    print(f"\n  → Cheapest: {rec['brand']} {rec['shade']} @ {rec['_best_price']} {rec['_best_retailer'].get('currency', '')}")
render_results(full_result)

In [ ]:
# Cell V3 — forced tradeoff (NYX cheap vs Charlotte Tilbury closest color)
tradeoff_set = [
    {"brand": "NYX",              "product": "Soft Matte Lip Cream",      "shade": "Monte Carlo",    "delta_e": 1.4},
    {"brand": "Charlotte Tilbury", "product": "Matte Revolution Lipstick", "shade": "Red Carpet Red", "delta_e": 0.3},
]
tradeoff_result = await run_tie_set(tradeoff_set, country="US")
print("V3 — tradeoff detected:", tradeoff_result["tradeoff"])
print("   alternative:", tradeoff_result["alternative"]["shade"] if tradeoff_result["alternative"] else "none")
render_results(tradeoff_result)

In [ ]:
# Cell V4 — country change (UK)
uk_result = await run_tie_set(TIE_SET[:2], country="UK")
print("V4 — UK retailers and currency:")
for c in uk_result["all_candidates"]:
    for p in c.get("prices", []):
        print(f"  {c['shade']}: {p.get('currency')} {p.get('price')} @ {p.get('retailer')}")

In [ ]:
# Cell V5 — no hardcoded secrets
import subprocess
out = subprocess.run(
    ["grep", "-rn", r"GOOGLE_API_KEY\s*=\|api_key\s*=",
     "agent_dupe_price_finder.ipynb", "adk_setup.py"],
    capture_output=True, text=True
)
hits = [l for l in out.stdout.splitlines() if "os.environ" not in l and "dotenv" not in l]
if hits:
    print("FAIL — hardcoded secrets found:")
    for h in hits:
        print(" ", h)
else:
    print("✓ V5 — no hardcoded secrets found (keys read from environment only)")

## 10. URL quality audit

`classify_url()` inspects every URL the agent returns and flags whether it's a real
product page, a search/category page the agent hallucinated, a Google grounding
redirect, or missing entirely.

Call `audit_prices(candidate["prices"])` on any candidate after a run to add a
`url_status` field to each price entry.

In [ ]:
# Cell 10 — URL quality audit (audit_prices + classify_url tests)


def audit_prices(prices: list[dict]) -> list[dict]:
    """Add a 'url_status' field to each price entry based on URL format."""
    return [{**p, "url_status": classify_url(p.get("url"))} for p in prices]


_test_urls = [
    ("https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHxxx",  "vertex_link"),
    ("https://www.macys.com/shop/product/mac-retro-matte-lipstick?ID=4517869",      "retailer_link"),
    ("https://www.gosupps.com/mac-macximal-silky-matte-ruby-woo",                   "retailer_link"),
    ("https://www.target.com/s?searchTerm=mac+ruby+woo+lipstick",                   "hallucinated"),
    ("https://www.ulta.com/search?query=nyx+monte+carlo",                            "hallucinated"),
    ("https://www.ulta.com",                                                          "hallucinated"),
    ("https://www.walmart.com",                                                       "hallucinated"),
    ("https://colourpop.com",                                                         "hallucinated"),
    (None,                                                                             "missing"),
]
for url, expected in _test_urls:
    got = classify_url(url)
    print(f"{'✓' if got == expected else '✗'} {expected:14s}  {str(url)[:70]}")
